# Wiki Scrape

a script to scrape star names off of wikipedia's list of stars, therefore assuring that data on pageviews can be found

In [48]:
## imports

import requests
from bs4 import BeautifulSoup
import random
import pandas as pd

# Data Extraction

In [56]:
## constants

HEADERS = {'User-Agent': 'CelestialCelebs (sw2499@cornell.edu)'}

In [ ]:
## a function to get data from wikipedia api
#
# params:
#   PAGE: the name of the Wikipedia page to fetch data for
#   FILE_PATH: the path to the file where the fetched data will be saved
# returns:
#   None

def get_wiki_data(page, file_path):
    r = requests.get(
        'https://en.wikipedia.org/w/api.php',
        params={'action': 'parse', 'format': 'json', 'page': page, 'prop': 'text', 'formatversion': 2},
        headers=HEADERS,
        timeout=60
    )
    r.raise_for_status()
    data = r.json()['parse']['text']

    # write to a file so we can inspect data without repeatedly scraping
    f = open(file_path, 'w')
    f.write(data)
    f.close()

In [ ]:
get_wiki_data('List_of_proper_names_of_stars', 'stars.html')
get_wiki_data('List_of_proper_names_of_exoplanets', 'exoplanets.html') # WIP
get_wiki_data('List_of_galaxies', 'galaxies.html') # WIP

# Data Parsing

In [ ]:
## read data

data = open('exoplanets.html', 'r').read()

## random seed

RANDOM_SEED = 8
random.seed(RANDOM_SEED)

In [81]:
## parse html data to find star names

soup = BeautifulSoup(data, 'html.parser') 
table = soup.select('table.wikitable')[0] # find data table on page
rows = table.find_all('tr') # in the table, find all rows
N = len(rows) # total number of rows

In [82]:
print(table.find_all('th'))

[<th>Name of planet
</th>, <th>Eponym
</th>, <th>Name of star
</th>, <th data-sort-type="number">Mass (<var>M</var><sub>J</sub>)
</th>, <th data-msort-type="number">Radius (<var>R</var><sub>J</sub>)
</th>, <th data-sort-type="number"><a href="/wiki/Orbital_period" title="Orbital period">Period</a> (days)
</th>, <th data-sort-type="number"><a href="/wiki/Semi-major_and_semi-minor_axes#Astronomy" title="Semi-major and semi-minor axes">Semi-major axis</a> (AU)
</th>, <th>Discovery Method
</th>, <th data-sort-type="number">Disc. year
</th>, <th data-sort-type="number">Distance (ly)
</th>, <th data-sort-type="number">Host star mass (<a href="/wiki/Solar_mass" title="Solar mass"><var>M</var><sub>☉</sub></a>)
</th>, <th data-sort-type="number"><a href="/wiki/Effective_temperature" title="Effective temperature">Host star temp.</a> (K)
</th>, <th>Remarks
</th>]


In [ ]:
## function to parse grid cells here

In [74]:
## a function to randomly select stars from the table, given that the star has a valid hyperlink to another page
#
# params:
#   n: number of random stars to select
# returns:
#   2D list containing [common_name, designation] for each selected star

def select_random_stars(n):
    objs = []
    row_idxs = random.sample(range(1, N), n)
    for idx in row_idxs:
        # get row from table
        row = rows[idx]
        # extract useful data from row
        [common_name, designation, _, _, _, _] = [td for td in row.find_all('td')]
        if ((common_name.a and 'new' not in common_name.find('a', href=True).get('class', [])) or 
            (designation.a and 'new' not in designation.find('a', href=True).get('class', []))): # cell is hyperlinked
            objs.append((common_name.get_text(strip=True), designation.get_text(strip=True)))
    return pd.DataFrame(objs, columns=['common_name', 'designation'])

In [ ]:
## TODO: figure out how to parse merged cells

## a function to randomly select exoplanets from the table, given that the exoplanet has a valid hyperlink to another page
#
# params:
#   n: number of random exoplanets to select
# returns:
#   list containing [common name, host star] for each selected exoplanet

def select_random_planets(n):
    objs = []
    row_idxs = random.sample(range(1, N), n)
    for idx in row_idxs:
        # get row from table
        row = rows[idx]
        # extract useful data from row
        cells = row.find_all('td')
        common_name = cells[0]
        host_star = cells[2]
        if (common_name.a and 'new' not in common_name.find('a', href=True).get('class', [])): # cell is hyperlinked
            objs.append((common_name.get_text(strip=True), host_star.get_text(strip=True)))
    return pd.DataFrame(objs, columns=['common_name', 'host_star'])

In [ ]:
## select 200 stars and save as a csv
# select_random_stars(200).to_csv('stars.csv', index=False)
# file exists now, don't need to run it again

In [ ]:
## select 75 exoplanets and save as a csv
# select_random_planets(75).to_csv('exoplanets.csv', index=False)
# function is broken lol